# LLM access  

In [292]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    base_url="http://194.171.191.226:3061",
    model="llama3.1:8b",
    format="json"
)

In [293]:
model.invoke("Give me a json answer of the question: Hi, How are you? }")

AIMessage(content='{\n  "response": {\n    "message": "I\'m doing well, thank you for asking!",\n    "tone": "friendly"\n  }\n}', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2025-04-16T15:17:09.864748377Z', 'done': True, 'done_reason': 'stop', 'total_duration': 367964036, 'load_duration': 42319951, 'prompt_eval_count': 26, 'prompt_eval_duration': 30302000, 'eval_count': 32, 'eval_duration': 252008000, 'model_name': 'llama3.1:8b'}, id='run-1300c8ce-c5c0-4cc8-a354-627c4f001fba-0', usage_metadata={'input_tokens': 26, 'output_tokens': 32, 'total_tokens': 58})

# Workflow

### State Of Workflow

In [294]:
from typing_extensions import TypedDict


class WorkflowState(TypedDict):
    question: str
    topic: str
    answer: str


### Creating Knowledge Context

In [295]:
from langchain_community.document_loaders import TextLoader


class ContextStore:
    def __init__(self):
        self.docs = None

    def load_context(self, path):
        loader = TextLoader(file_path=path)
        docs = loader.load()

        self.docs = docs

    def get_context(self):
        return self.docs

In [296]:
activities_store = ContextStore()
company_store = ContextStore()
esta_store = ContextStore()
eta_store = ContextStore()
packlist_store = ContextStore()
a_to_z_store = ContextStore()
spa_store = ContextStore()
wifi_store = ContextStore()

activities_store.load_context(path="splitted/activities.txt")
company_store.load_context(path="splitted/company.txt")
esta_store.load_context(path="splitted/esta.txt")
eta_store.load_context(path="splitted/eta.txt")
packlist_store.load_context(path="splitted/packlist.txt")
a_to_z_store.load_context(path="splitted/scenic-eclipse-a-z")
spa_store.load_context(path="splitted/spa.txt")
wifi_store.load_context(path="splitted/wifi.txt")

stores = {
    "activities": activities_store,
    "company": company_store,
    "esta": esta_store,
    "eta": eta_store,
    "packlist": packlist_store,
    "scenic-eclipse-a-z": a_to_z_store,
    "spa": spa_store,
    "wifi": wifi_store
}

### Tools

- Retrieve related documents
- Check if documents relevant
- Generate answer
- Check if answer is relevant

In [297]:
from langchain_core.prompts import ChatPromptTemplate
import json

extract_topic_prompt = ChatPromptTemplate.from_messages([
    ("user", """
    
    You are an AI router in a RAG pipeline. You are given with CATEGORIES of possible routes and MESSAGE of input. You need to output the category, that might be related to a certain CATEGORY.
    
    CATEGORIES and description.
    
    activities: current tours and activities available in the program
    company: general information about the company
    esta: guide to apply for the ESTA visa for entry into the USA
    eta: guide to apply for the ETA visa for entry into Canada
    packlist: what items to take in the trip
    scenic-eclipse-a-z: A list of all little details from A - Z about the ship.
    spa: Brouchure about services and prices in SPA
    wifi: guide how to connect to the local wifi
    
    If the MESSAGE is not related to any topic, output "other"
    
    Generate only one value. The value can be only from the list [activities, company, esta, eta, packlist, scenic-eclipse-a-z, spa, wifi, other]
    
    Output has to be a valid json:
    result: SELECTED_VALUE
    
    Don't provide any other information
    
    MESSAGE: {question}. 
    """)
])


def extract_topic(state: WorkflowState):
    messages = extract_topic_prompt.invoke({"question": state["question"]})
    res = model.invoke(messages).content

    res = json.loads(res)["result"]

    return {"topic": res}

In [298]:
def general_qa_router(state: WorkflowState):
    if state["topic"] == "other":
        return "generate_general_answer"
    return "generate_qa_answer"

In [299]:
generate_general_answer_prompt = ChatPromptTemplate.from_messages([
    ("user", """
    MESSAGE: {question}.
        
    You are an AI agent that is included in AI support of clients. You need to answer the given MESSAGE that you cannot help with that. You can output only valid json with 1 key:
    answer: string
    
    Your answer has to be concise, formal style and professional. 
    Your answer has to contain only the information, that you cannot help with that.
    Your answer has to contain a question about any other help on the following topics, but to don write them directly:  
    
    activities: current tours and activities available in the program
    company: general information about the company
    esta: guide to apply for the ESTA visa for entry into the USA
    eta: guide to apply for the ETA visa for entry into Canada
    packlist: what items to take in the trip
    scenic-eclipse-a-z: A list of all little details from A to Z about the ship.
    spa: Brouchure about services and prices in SPA
    wifi: guide how to connect to the local wifi
    """)
])

def generate_general_answer(state: WorkflowState):
    prompt = generate_general_answer_prompt.invoke({"question": state["question"]})
    res = model.invoke(prompt).content

    res = json.loads(res)["answer"]
    
    return {"answer": res}

In [300]:
generate_qa_answer_prompt = ChatPromptTemplate.from_messages([
    ("user", """
    CONTEXT: \n\n {context} \n\n MESSAGE: {question}.
        
    You are an AI agent that is included in AI support of clients. You need to answer the given MESSAGE based on given CONTEXT. You can output only valid json with 1 key:
    answer: string
    
    Your answer has to be formal style and professional.
    
    Your answer must contain information only from provided CONTEXT. 
    
    Your answer must be helpful.
    
    Your answer should not refer to the CONTEXT
    
    At the end you need to propose any other help.
    
    Do not try to come up with an answer if the MESSAGE does not contain an answer from CONTEXT. 
    """)
])


def generate_qa_answer(state: WorkflowState):
    store = stores[state["topic"]]
    docs = store.search()

    prompt = generate_qa_answer_prompt.invoke({"question": state["question"], "context": docs})
    res = model.invoke(prompt).content

    res = json.loads(res)["answer"]

    return {"answer": res}


In [301]:
from langgraph.graph import START, StateGraph, END

graph_builder = StateGraph(WorkflowState)

graph_builder.add_node(extract_topic, "extract_topic")
graph_builder.add_node(generate_general_answer, "generate_general_answer")
graph_builder.add_node(generate_qa_answer, "generate_qa_answer")

graph_builder.add_edge(START, "extract_topic")

graph_builder.add_conditional_edges("extract_topic", general_qa_router)

graph_builder.add_edge("generate_general_answer", END)
graph_builder.add_edge("generate_qa_answer", END)

graph = graph_builder.compile()

In [334]:
graph.invoke({"question": "How much does it cost?"})

{'question': 'How much does it cost?',
 'topic': 'spa',
 'answer': "We do not have a specific price listed for 'How much does it cost?' in our menu. However, we can provide you with more information on our services and prices if you'd like to discuss further."}